##Install Necessary Dependencies
Installs a set of Python libraries commonly used for working with Large Language Models (LLMs), data processing, and visualization.

In [ ]:
# ── CELL 1: Install ───────────────────────────────────────────
!pip install -q transformers accelerate torch plotly pandas requests pyyaml bitsandbytes
!pip install -U bitsandbytes>=0.46.1

## LLM Selection, Load Files, Scoring Logic
- Load all 3 csv files
- Load of 3 configuration files



In [ ]:
# ── CELL 2: Backend config ─────────────────────────────────────
import os

# HuggingFace is the LLM backend for this project
LLM_BACKEND  = "huggingface"
HF_MODEL_ID  = "Qwen/Qwen2.5-3B-Instruct"
HF_TOKEN     = os.environ.get("HF_TOKEN", "")

OWNERSHIP_DEPTH_PCT = 1   # flag any stake >= this %

# ── CELL 3: Country risk baseline & SOE helpers ────────────────
import re

COUNTRY_RISK_BASELINE = {
    "Russia": 55, "China": 45, "Iran": 50, "Syria": 58,
    "France": 28, "Germany": 8, "UK": 7, "Israel": 20,
    "South Africa": 14, "Austria": 10, "Spain": 9, "Singapore": 5,
    "India": 6, "USA": 4, "Canada": 4,
    "South Korea": 9, "Poland": 12,
    "Belgium": 9, "Sweden": 6, "Norway": 5,
}

SOE_KEYWORDS = re.compile(
    r"government|sovereign|state|gic|temasek|bpifrance|lic of india|soe|"
    r"public sector|national investment|pension fund|insurer", re.I)
STATE_TYPES = {"Government", "Sovereign Wealth Fund", "Government Investment",
               "Government Pension Fund", "Government Insurer", "SOE",
               "Industrial Conglomerate", "Public Sector"}

_ALLIED  = {"India", "Russia", "Israel", "France", "United States", "USA",
            "UK", "United Kingdom", "Japan", "South Korea", "Australia"}
_NEUTRAL = {"Germany", "Italy", "Sweden", "Spain", "Brazil", "Singapore",
            "Malaysia", "Turkey", "UAE", "South Africa"}

def country_risk(name: str) -> float:
    return COUNTRY_RISK_BASELINE.get(str(name).strip(), 30.0)

# ── CELL 4: CSV file helpers ───────────────────────────────────
import pandas as pd

ALIAS_MAP: dict = {"Ford Motor Co.": "Ford Motor Company",
}   # add spelling aliases here if needed

def normalize_name(s) -> str:
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    s = str(s).strip()
    return ALIAS_MAP.get(s, s).strip().lower()


COUNTRY_ALIASES = {
    "usa": "USA", "us": "USA", "u.s.": "USA", "u.s.a.": "USA",
    "united states": "USA", "united states of america": "USA", "america": "USA",
    "uk": "UK", "u.k.": "UK", "united kingdom": "UK", "great britain": "UK", "britain": "UK",
    "russia": "Russia", "russian federation": "Russia",
    # Add more spelling aliases here as new countries show up in your data
}

def normalize_country(name) -> str:
    """Canonicalises a country string for matching purposes (lower-cased,
    trimmed, alias-resolved). Used identically for every country in the
    dataset — there is no per-country special-casing anywhere downstream."""
    if name is None or (isinstance(name, float) and pd.isna(name)):
        return ""
    s = str(name).strip()
    if not s or s.lower() in ("unknown", "nan", "multiple", "—", "-", "none"):
        return ""
    key = s.lower()
    canonical = COUNTRY_ALIASES.get(key, s)
    return canonical.strip().lower()

from google.colab import files

# Stores uploaded filenames
_UPLOADED_FILES = None

def upload_all_csvs():
    """Upload all required CSV files in a single file picker."""
    global _UPLOADED_FILES

    if _UPLOADED_FILES is None:
        print("⬆️ Upload these files together:")
        print("   • supply_chain.csv")
        print("   • ownership.csv")
        print("   • indian_alternates.csv")
        _UPLOADED_FILES = files.upload()

    return _UPLOADED_FILES

def _get_local_or_upload(path_hint: str, filename: str) -> str:
    """Use local file if present, otherwise use the batch upload."""

    # Prefer local copy if it already exists
    if os.path.exists(path_hint):
        return path_hint

    uploaded = upload_all_csvs()

    if filename in uploaded:
        return filename

    raise FileNotFoundError(
        f"{filename} was not included in the upload."
    )

def _read_csv_robust(path: str, label: str) -> pd.DataFrame:
    for enc in ["utf-8", "utf-8-sig", "cp1252", "latin-1"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            if enc != "utf-8":
                print(f"ℹ️  {label}: read using '{enc}' encoding.")
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Could not decode {label} with any supported encoding.")

# ── Column-name normalisation ──────────────────────────────────
_COLUMN_ALIASES = {
    "supplier":       {"supplier", "suppliers",
                        "supplier_name", "company", "company_name"},
    "owner":          {"owner", "owners", "owner_name"},
    "owner_type":     {"owner_type", "ownertype", "type"},
    "owner_country":  {"owner_country", "ownercountry", "country"},
    "stake_pct":      {"stake_pct", "stake_percent", "stake_percentage",
                        "stake", "pct", "ownership_pct"},
    "board_role":     {"board_role", "boardrole", "role"},
    "vehicle":        {"vehicle", "vehicles"},
    "component":      {"component", "components"},
    "category":       {"category", "categories"},
    "tier1_country":  {"tier1_country", "tier1country", "tier_1_country"},
    "tier2_supplier": {"tier2_supplier", "tier2supplier", "tier_2_supplier"},
    "tier2_country":  {"tier2_country", "tier2country", "tier_2_country"},
}

def _normalize_columns(df: pd.DataFrame, label: str) -> pd.DataFrame:
    """Renames df columns to canonical names wherever a known alias is
    found (case/whitespace-insensitive). Unmatched columns are untouched."""
    alias_to_canonical = {}
    for canonical, aliases in _COLUMN_ALIASES.items():
        for a in aliases:
            alias_to_canonical[a] = canonical

    rename_map = {}
    for col in df.columns:
        key = str(col).strip().lower().replace(" ", "_")
        canonical = alias_to_canonical.get(key)
        if canonical and canonical != col:
            rename_map[col] = canonical

    if rename_map:
        print(f"ℹ️  {label}: normalised columns {rename_map}")
        df = df.rename(columns=rename_map)
    return df

# ── CELL 5: Load supply_chain.csv ─────────────────────────────
def load_supply_chain_csv(path_hint="/content/supply_chain.csv") -> pd.DataFrame:
    fp   = _get_local_or_upload(path_hint, "supply_chain.csv")
    df   = _read_csv_robust(fp, "supply_chain.csv")
    df   = _normalize_columns(df, "supply_chain.csv")
    # supply_chain.csv's tier-1 supplier column shares the alias set with
    if "supplier" in df.columns and "tier1_supplier" not in df.columns:
        df = df.rename(columns={"supplier": "tier1_supplier"})
    expected = {"vehicle", "component", "tier1_supplier", "tier1_country",
                "tier2_supplier", "tier2_country"}
    missing  = expected - set(df.columns)
    if missing:
        print(f"⚠️  supply_chain.csv missing expected columns: {missing}")
    print(f"✅ supply_chain.csv: {len(df)} rows")
    return df

# ── CELL 6: Load ownership.csv ────────────────────────────────
def load_ownership_csv(path_hint="/content/ownership.csv") -> pd.DataFrame:
    fp  = _get_local_or_upload(path_hint, "ownership.csv")
    df  = _read_csv_robust(fp, "ownership.csv")
    df  = _normalize_columns(df, "ownership.csv")
    expected = {"supplier", "owner", "owner_type", "owner_country",
                "stake_pct", "board_role", "financial_distress"}
    missing  = expected - set(df.columns)
    if missing:
        print(f"⚠️  ownership.csv missing expected columns: {missing}")
    print(f"✅ ownership.csv: {len(df)} rows")
    return df

# ── CELL 7: Load indian_alternates.csv ────────────────────────
def load_indian_alternates_csv(path_hint="/content/indian_alternates.csv") -> dict:
    fp  = _get_local_or_upload(path_hint, "indian_alternates.csv")
    df  = _read_csv_robust(fp, "indian_alternates.csv")
    catalog: dict = {}
    for _, r in df.iterrows():
        name = r.get("company_name", "")
        if pd.isna(name) or not str(name).strip():
            continue
        cat = r.get("component_category", "default")
        cat = "default" if pd.isna(cat) or not str(cat).strip() else str(cat).strip().lower()
        comps   = r.get("components_supplied", "")
        comps   = "" if pd.isna(comps) else str(comps).strip()
        country = r.get("country", "India")
        country = "India" if pd.isna(country) or not str(country).strip() else str(country).strip()
        lead_time = r.get("lead_time_weeks", None)
        try:
            lead_time = float(lead_time) if lead_time is not None and not pd.isna(lead_time) else None
        except (TypeError, ValueError):
            lead_time = None
        catalog.setdefault(cat, []).append({
            "name": str(name).strip(), "components_supplied": comps,
            "country": country, "component_category": cat,
            "lead_time_weeks": lead_time,
        })
    if not catalog.get("default"):
        pool = [r for recs in catalog.values() for r in recs]
        catalog["default"] = pool[:5] or [{
            "name": "Bharat Forge Ltd", "components_supplied": "General defense components",
            "country": "India", "component_category": "default",
        }]
    print(f"✅ indian_alternates.csv: {len(df)} records across {len(catalog)} categories")
    return catalog

# ── CELL 8: Load all data ──────────────────────────────────────
supply_df  = load_supply_chain_csv()
owner_df   = load_ownership_csv()
INDIAN_ALTERNATES_CATALOG = load_indian_alternates_csv()

# Build ownership registry keyed by NORMALISED supplier name
OWNERSHIP_REGISTRY: dict = {}
for _, row in owner_df.iterrows():
    key = normalize_name(row.get("supplier"))
    if not key:
        continue
    stake = row.get("stake_pct")
    try:
        stake = float(stake) if not pd.isna(stake) else 0.0
    except (TypeError, ValueError):
        stake = 0.0
    board = row.get("board_role", "")
    # financial_distress column in ownership.csv: 0 = financially healthy,
    # 5 = moderate distress signals, 10 = severe distress. Optional — if
    # absent, this term of the risk formula simply contributes 0.
    fin = row.get("financial_distress", None)
    try:
        fin = float(fin) if fin is not None and not pd.isna(fin) else None
    except (TypeError, ValueError):
        fin = None
    OWNERSHIP_REGISTRY.setdefault(key, []).append({
        "owner":      row.get("owner", "Unknown"),
        "type":       row.get("owner_type", "Unknown"),
        "country":    row.get("owner_country", "Unknown"),
        "stake_pct":  stake,
        "board_role": "" if pd.isna(board) else str(board),
        "financial_distress": fin,
    })
print(f"✅ Ownership registry: {len(OWNERSHIP_REGISTRY)} unique supplier names")

FINANCIAL_DISTRESS: dict = {}
for _key, _owners in OWNERSHIP_REGISTRY.items():
    _vals = [o["financial_distress"] for o in _owners
             if o.get("financial_distress") is not None]
    FINANCIAL_DISTRESS[_key] = max(_vals) if _vals else 0.0

def financial_distress_score(supplier_name: str) -> float:
    """0–10 pts — the 'Financial Distress Score' term of the risk formula.
    Reads the `financial_distress` column from ownership.csv (per-owner;
    the supplier takes its MOST distressed owner's value, since one
    financially-troubled parent/owner is enough to put the whole
    supplier at risk). Defaults to 0 for any supplier with no such data."""
    return min(10.0, max(0.0, FINANCIAL_DISTRESS.get(normalize_name(supplier_name), 0.0)))

# Build supply-chain edge list (both tiers)
GRAPH: list = []
for _, r in supply_df.iterrows():
    t2_sup   = r.get("tier2_supplier", r.get("tier1_supplier", ""))
    t2_ctry  = r.get("tier2_country",  r.get("tier1_country", "Unknown"))
    comp     = r.get("component", r.get("tier2_component", r.get("function", "")))
    GRAPH.append({
        "vehicle":   str(r.get("vehicle",       "")),
        "category":  str(r.get("category",      "")),
        "component": str(comp),
        "tier1":     str(r.get("tier1_supplier","")),
        "t1_country":str(r.get("tier1_country", "Unknown")),
        "tier2":     str(t2_sup),
        "t2_country":str(t2_ctry),
    })
print(f"✅ Supply chain GRAPH: {len(GRAPH)} edges")

# ── Concentration map — how many distinct suppliers provide a specific component
CONCENTRATION_MAP: dict = {}
for edge in GRAPH:
    key = (edge["vehicle"], edge["component"])
    for sup in (edge["tier1"], edge["tier2"]):
        if sup:
            CONCENTRATION_MAP.setdefault(key, set()).add(sup)

def concentration_penalty(supplier_name: str) -> float:
    """0–5 pts — the 'Concentration Penalty' term of the risk formula.
    Highest when the supplier is the ONLY source for one or more
    vehicle/component combinations it's involved in."""
    combos = [k for k, sups in CONCENTRATION_MAP.items() if supplier_name in sups]
    if not combos:
        return 0.0
    sole_source_combos = sum(1 for k in combos if len(CONCENTRATION_MAP[k]) == 1)
    if sole_source_combos == 0:
        return 0.0
    ratio = sole_source_combos / len(combos)
    return round(min(5.0, 2.0 + 3.0 * ratio), 1)

# ── Tier-1 → Tier-2 country dependency map ─────────────────────
TIER1_TO_TIER2_COUNTRIES: dict = {}
for edge in GRAPH:
    if edge["tier1"] and edge["tier2"] and edge["tier1"] != edge["tier2"]:
        TIER1_TO_TIER2_COUNTRIES.setdefault(edge["tier1"], set()).add(
            (edge["tier2"], edge["t2_country"]))

# Diagnostic: warn about suppliers with no ownership data
all_supplier_names = set()
for edge in GRAPH:
    all_supplier_names.update([edge["tier1"], edge["tier2"]])
all_supplier_names.discard("")
unmatched = [n for n in sorted(all_supplier_names) if normalize_name(n) not in OWNERSHIP_REGISTRY]
if unmatched:
    print(f"⚠️  {len(unmatched)} suppliers have no ownership record (name mismatch or genuine gap):")
    for n in unmatched:
        print(f"   - {n}")
else:
    print("✅ Every supplier has at least one ownership record.")

SUPPLIER_COUNT = len(all_supplier_names)

# ── CELL 9: Supply agent configuration files ──
import yaml, json as _json

CREAO_CONFIG    = None
CREAO_MANIFEST  = None
CREAO_SKILL     = None

required_files = [
    "config.yaml",
    "manifest.json",
    "SKILL.md"
]

missing = [f for f in required_files if not os.path.exists(f"/content/{f}")]

if missing:
    try:
        from google.colab import files

        print("📂 Supply agent configuration files")
        print("Upload any or all of the following in ONE upload dialog:")
        for f in required_files:
            print(f"   • {f}")
        print("\n(config.yaml drives live parameters: ownership_depth, "
              "risk_threshold, alternate_supplier_count.)")
        print("Or press Cancel to use built-in defaults.\n")

        uploaded = files.upload()

    except Exception:
        uploaded = {}
else:
    uploaded = {}

# ------------------------------------------------------------

def load_if_exists(filename, loader):
    path = f"/content/{filename}"

    # Uploaded files are saved in current working directory.
    # Prefer uploaded copy if present.
    if os.path.exists(filename):
        path = filename

    if not os.path.exists(path):
        return None

    try:
        with open(path, encoding="utf-8") as f:
            data = loader(f)
        print(f"✅ Loaded {filename}")
        return data
    except Exception as e:
        print(f"⚠️ Could not parse {filename}: {e}")
        return None

# ------------------------------------------------------------

CREAO_CONFIG = load_if_exists("config.yaml", yaml.safe_load)
CREAO_MANIFEST = load_if_exists("manifest.json", _json.load)
CREAO_SKILL = load_if_exists("SKILL.md", lambda f: f.read())

# ------------------------------------------------------------

AGENT_TITLE = (CREAO_MANIFEST or {}).get(
    "name",
    "Indian Army Supply Chain Risk Monitor"
)

# ------------------------------------------------------------
# Turn config.yaml's form-field defaults into real, live pipeline
# parameters.
# ------------------------------------------------------------

_DEPTH_MAP = {
    "one_percent": 1,
    "five_percent": 5,
    "ten_percent": 10,
    "twenty_five_percent": 25,
}

# risk_threshold controls the numeric cutoff used for the "Alert
# Threshold" line on the risk chart AND the deterministic "Critical
# Alerts" KPI count — i.e. it actually changes what counts as an
# alert, not just a label.
_RISK_THRESHOLD_MAP = {
    "critical": 85,
    "high":     70,
    "medium":   50,
    "all":      0,
}

# Defaults if config.yaml is absent or a field is missing.
RISK_ALERT_THRESHOLD = 70
ALT_SUPPLIER_COUNT   = 3

def _config_field(config, field_id):
    """Return the raw config.yaml field dict for a given form field id."""
    if not config:
        return None
    for field in config.get("form", {}).get("fields", []):
        if field.get("id") == field_id:
            return field
    return None

_own_field = _config_field(CREAO_CONFIG, "ownership_depth")
if _own_field is not None:
    OWNERSHIP_DEPTH_PCT = _DEPTH_MAP.get(
        _own_field.get("default_value"), OWNERSHIP_DEPTH_PCT
    )

_risk_field = _config_field(CREAO_CONFIG, "risk_threshold")
if _risk_field is not None:
    RISK_ALERT_THRESHOLD = _RISK_THRESHOLD_MAP.get(
        _risk_field.get("default_value"), RISK_ALERT_THRESHOLD
    )

_alt_field = _config_field(CREAO_CONFIG, "alternate_supplier_count")
if _alt_field is not None:
    try:
        ALT_SUPPLIER_COUNT = int(_alt_field.get("default_value", ALT_SUPPLIER_COUNT))
    except (TypeError, ValueError):
        pass

if CREAO_SKILL:
    print(f"📖 SKILL.md loaded ({len(CREAO_SKILL)} chars)")

print(
    f"\nGRAPH edges: {len(GRAPH)}"
    f"  |  Suppliers: {SUPPLIER_COUNT}"
    f"  |  Ownership depth: >={OWNERSHIP_DEPTH_PCT}%"
    f"  |  Alert threshold: >={RISK_ALERT_THRESHOLD} (from risk_threshold config)"
    f"  |  Alts/supplier: {ALT_SUPPLIER_COUNT}"
    f"  |  LLM: {LLM_BACKEND.upper()}"
)

# ── CELL 10: Deterministic SOE / ownership analysis ───────────
def run_ownership_analysis(supplier_names, depth_pct=1.0) -> list:
    results = []
    seen_names = set()
    for name in supplier_names:
        # de-dupe defensively: if the same supplier name appears twice in
        if name in seen_names:
            continue
        seen_names.add(name)
        owners = [o for o in OWNERSHIP_REGISTRY.get(normalize_name(name), [])
                  if o["stake_pct"] >= depth_pct]
        state_owners = [o for o in owners
                        if o["type"] in STATE_TYPES
                        or SOE_KEYWORDS.search(str(o["type"]))
                        or SOE_KEYWORDS.search(str(o["owner"]))]
        total_state_pct = min(sum(o["stake_pct"] for o in state_owners), 100)
        soe_flag = (
            any(o["stake_pct"] > 25 and (o["type"] in STATE_TYPES or SOE_KEYWORDS.search(str(o["type"])))
                for o in owners)
            or total_state_pct > 40
        )
        results.append({
            "name": name, "owners": owners, "state_owners": state_owners,
            "total_state_pct": round(total_state_pct, 1), "soe_flag": soe_flag,
            "ownership_risk_score": min(20, int(total_state_pct / 5)),
        })
    return results

# ── CELL 11: Build supplier roll-up for LLM context ───────────
def build_supplier_rollup() -> list:
    rollup = []
    for name in sorted(all_supplier_names):
        edges      = [e for e in GRAPH if e["tier1"] == name or e["tier2"] == name]
        countries  = sorted({e["t1_country"] if e["tier1"] == name else e["t2_country"] for e in edges})
        vehicles   = sorted({e["vehicle"]   for e in edges if e["vehicle"]})
        components = sorted({e["component"] for e in edges if e["component"]})
        tiers      = sorted({"Tier-1" if e["tier1"] == name else "Tier-2" for e in edges})
        soe_data   = next((r for r in run_ownership_analysis([name]) if r["name"] == name), {})
        base_risk  = max([country_risk(c) for c in countries], default=30.0)
        rollup.append({
            "name": name, "countries": countries, "vehicles": vehicles,
            "components": components, "tiers": tiers,
            "soe_flag":       soe_data.get("soe_flag", False),
            "total_state_pct":soe_data.get("total_state_pct", 0.0),
            "baseline_country_risk": base_risk,
        })
    return rollup

supplier_rollup = build_supplier_rollup()
print(f"✅ Roll-up built for {len(supplier_rollup)} suppliers | "
      f"{sum(1 for s in supplier_rollup if s['soe_flag'])} state-linked/controlled")

# ── CELL 12: Indian alternate lookup helpers ───────────────────
def _indian_alternate(component: str, supplier_name: str) -> dict:
    comp_lower = component.lower()
    cat = next((c for c in INDIAN_ALTERNATES_CATALOG if c != "default" and c in comp_lower), "default")
    options = INDIAN_ALTERNATES_CATALOG.get(cat) or INDIAN_ALTERNATES_CATALOG["default"]
    return options[sum(ord(c) for c in supplier_name) % len(options)]

def readiness_score(alt: dict, s: dict) -> dict:
    #Deterministic Alternate Supplier Readiness Score
    comp_lower = str(s.get("component", "")).lower()
    alt_cat    = str(alt.get("component_category", "")).lower()
    alt_comps  = str(alt.get("components_supplied", "")).lower()
    if alt_cat and alt_cat in comp_lower:
        capability_match = 30
    elif any(word in alt_comps for word in comp_lower.split() if len(word) > 3):
        capability_match = 15
    elif alt_cat and alt_cat != "default":
        capability_match = 5
    else:
        capability_match = 0

    # -- Country Risk Bonus (0-25): India catalog is mostly constant,
    #    keep the branch for future non-Indian alternates
    alt_country    = str(alt.get("country", "")).strip()
    crisis_country = str(s.get("t1_country", s.get("country", ""))).strip()
    if alt_country.lower() == "india":
        country_risk_bonus = 25
    elif alt_country and alt_country.lower() != crisis_country.lower():
        country_risk_bonus = 12   # allied/neutral, non-crisis country
    else:
        country_risk_bonus = 0    # same country as the crisis-hit supplier

    # -- Capacity Signal (0-20): PROXY — number of components_supplied listed.
    n_comps = len([c for c in alt_comps.split(",") if c.strip()]) if alt_comps else 0
    capacity_signal = min(20, n_comps * 4)

    # -- Lead-Time Score (0-15):
    lt_weeks = alt.get("lead_time_weeks")
    if lt_weeks is not None:
        lead_time_score = max(0, min(15, round(15 - (lt_weeks / 16) * 15)))
        lead_time_is_estimated = False
    else:
        lead_time_score = 8
        lead_time_is_estimated = True

    # -- Existing Relationship (0-10): alt already supplies a different
    #    component in the SAME vehicle program -> faster onboarding
    existing_relationship = 0
    alt_name_norm = normalize_name(alt.get("name", ""))
    for edge in GRAPH:
        if edge["vehicle"] == s.get("vehicle") and (
            normalize_name(edge["tier1"]) == alt_name_norm or
            normalize_name(edge["tier2"]) == alt_name_norm
        ):
            existing_relationship = 10
            break

    total = min(100, capability_match + country_risk_bonus + capacity_signal
                + lead_time_score + existing_relationship)

    return {
        "readiness_score": total,
        "capability_match": capability_match,
        "country_risk_bonus": country_risk_bonus,
        "capacity_signal": capacity_signal,
        "lead_time_score": lead_time_score,
        "lead_time_is_estimated": lead_time_is_estimated,
        "existing_relationship": existing_relationship,
    }

##Download LLM

In [ ]:
# ── CELL 13a: Load HF model weights (run ONCE per runtime) ────
# downloads (first time) and loads the model weights into RAM/VRAM.

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

if LLM_BACKEND == "huggingface":
    _already_loaded = (
        "_HF_MODEL" in globals()
        and "_HF_TOKENIZER" in globals()
        and globals().get("_HF_MODEL_ID_LOADED") == HF_MODEL_ID
    )
    if _already_loaded:
        print(f"✅ {HF_MODEL_ID} already loaded in memory — skipping reload.")
    else:
        _device_ok = torch.cuda.is_available()
        print(f"🤗 Loading: {HF_MODEL_ID}")
        print(f"   Device: {'GPU ✅' if _device_ok else 'CPU ⚠️'}")

        if not _device_ok:
            print("   ⚠️  NO GPU DETECTED. A 3B model on CPU can take 10-30+ minutes")
            print("      PER LLM CALL — this is almost certainly why Cells 13b/14/")
            print("      15/16/18/19 looked stuck. Strongly recommended before")
            print("      continuing: Runtime > Change runtime type > T4 GPU, then")
            print("      Runtime > Restart session, and re-run from Cell 1.")

        _HF_TOKENIZER = AutoTokenizer.from_pretrained(HF_MODEL_ID, token=HF_TOKEN or None)
        if _HF_TOKENIZER.pad_token is None:
            _HF_TOKENIZER.pad_token = _HF_TOKENIZER.eos_token

        # 4-bit quantization on GPU: ~4x less VRAM, meaningfully faster
        # generation, negligible quality loss for structured JSON output.
        _quant_cfg = None
        if _device_ok:
            try:
                from transformers import BitsAndBytesConfig
                _quant_cfg = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_compute_dtype=torch.float16,
                    bnb_4bit_use_double_quant=True,
                    bnb_4bit_quant_type="nf4",
                )
            except Exception as e:
                print(f"   ℹ️  4-bit quantization unavailable ({e}); loading full precision.")
                _quant_cfg = None

        _HF_MODEL = AutoModelForCausalLM.from_pretrained(
            HF_MODEL_ID,
            torch_dtype=torch.float16 if _device_ok else torch.float32,
            device_map="auto" if _device_ok else None,
            quantization_config=_quant_cfg,
            token=HF_TOKEN or None,
        )
        if not _device_ok:
            _HF_MODEL = _HF_MODEL.to("cpu")
        _HF_MODEL_ID_LOADED = HF_MODEL_ID
        print(f"✅ Model + tokenizer loaded{' (4-bit quantized)' if _quant_cfg else ''}.")
else:
    print(f"ℹ️  LLM_BACKEND is '{LLM_BACKEND}', not 'huggingface' — skipping local model load.")


##Pipeline Execution and Analysis

In [ ]:
# ── CELL 13b: Build the LLM client (lightweight — safe to re-run) ──
# This just wires up a `call_hf` function around the ALREADY-LOADED
import json as _json, textwrap, ast, warnings, datetime, time, threading
warnings.filterwarnings("ignore")

GENERATION_TIMEOUT_SEC = 180   # hard cap per LLM call, in seconds

def _repair_truncated_json(text: str) -> str:
    if not text:
        return "{}"
    text = text.rstrip()
    if text.endswith(","):
        text = text[:-1]
    open_braces   = text.count("{") - text.count("}")
    open_brackets = text.count("[") - text.count("]")
    return text + "]" * max(open_brackets, 0) + "}" * max(open_braces, 0)

def _extract_json(text: str) -> str:
    import re as _re
    text = text.strip()
    text = _re.sub(r"```(?:json)?", "", text).replace("```", "").strip()
    text = _re.sub(r"<\|.*?\|>", "", text).strip()
    start = text.find("{")
    if start == -1:
        return "{}"
    depth = 0
    for i, ch in enumerate(text[start:], start):
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start:i + 1]
    return _repair_truncated_json(text[start:])

def _parse_llm_output(raw: str) -> dict:
    cleaned = _extract_json(raw)
    parsed = {}
    for fn in [_json.loads,
               lambda s: _json.loads(s.replace("True","true").replace("False","false").replace("None","null")),
               ast.literal_eval]:
        try:
            parsed = fn(cleaned)
            break
        except Exception:
            pass
    if not isinstance(parsed, dict):
        return {}

    # Small local models occasionally drift on the exact key name/casing
    if "suppliers" not in parsed:
        for alt_key in ("Suppliers", "supplier_list", "supplier_scores", "results"):
            if alt_key in parsed and isinstance(parsed[alt_key], list):
                parsed["suppliers"] = parsed.pop(alt_key)
                break
    if "alternate_suppliers" not in parsed:
        for alt_key in ("Alternate_suppliers", "alternates", "alternate_supplier_list"):
            if alt_key in parsed and isinstance(parsed[alt_key], list):
                parsed["alternate_suppliers"] = parsed.pop(alt_key)
                break
    return parsed

def build_llm_client():
    if LLM_BACKEND == "huggingface":
        from transformers import TextIteratorStreamer

        def call_hf(system_msg: str, user_msg: str, max_new_tokens: int = 900) -> str:
            prompt = _HF_TOKENIZER.apply_chat_template(
                [{"role": "system", "content": system_msg},
                 {"role": "user",   "content": user_msg}],
                tokenize=False, add_generation_prompt=True,
            )
            inputs = _HF_TOKENIZER(prompt, return_tensors="pt").to(_HF_MODEL.device)
            streamer = TextIteratorStreamer(
                _HF_TOKENIZER, skip_prompt=True, skip_special_tokens=True
            )
            gen_kwargs = dict(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,          # greedy — far more reliable for JSON
                temperature=None,
                top_p=None,
                repetition_penalty=1.1,
                pad_token_id=_HF_TOKENIZER.eos_token_id,
                streamer=streamer,
            )

            gen_thread = threading.Thread(target=_HF_MODEL.generate, kwargs=gen_kwargs)
            gen_thread.start()

            chunks, start, last_heartbeat = [], time.time(), time.time()
            for piece in streamer:
                chunks.append(piece)
                now = time.time()
                if now - last_heartbeat > 5:
                    print(f"    ⏳ generating... {len(''.join(chunks))} chars, "
                          f"{now - start:.0f}s elapsed")
                    last_heartbeat = now
                if now - start > GENERATION_TIMEOUT_SEC:
                    print(f"    ⏱️  Hit {GENERATION_TIMEOUT_SEC}s timeout — "
                          f"using partial output instead of waiting longer.")
                    break
            gen_thread.join(timeout=1)   # don't block on background cleanup

            raw = "".join(chunks)
            print(f"  🔍 LLM raw (first 300 chars): {raw[:300]}")
            return raw

        print("✅ HuggingFace client ready (streaming + "
              f"{GENERATION_TIMEOUT_SEC}s timeout enabled)")
        return call_hf

    raise ValueError(
        f"LLM_BACKEND={LLM_BACKEND!r} is not supported. This project runs "
        "HuggingFace only — set LLM_BACKEND = 'huggingface' in Cell 2."
    )

print("\n🔄 Initialising LLM backend...")
LLM_FN = build_llm_client()

# ── CELL 14: LLM scenario analysis  ─────────────

SUPPLIER_BATCH_SIZE = 8

def _rollup_line(s: dict) -> str:
    return (f"  {s['name']} | tiers={s['tiers']} | countries={s['countries']} | "
            f"baseline_country_risk={s['baseline_country_risk']} | "
            f"soe_flag={s['soe_flag']} (state_stake={s['total_state_pct']}%) | "
            f"components={s['components']} | vehicles={s['vehicles']}")

def _framing_call(llm_fn, scenario: str, rollup: list) -> dict:
    """Small/fast call for just the narrative fields — uses compact
    name+country pairs only, not the full roll-up, to keep it short."""
    names_countries = "\n".join(f"  {s['name']} ({', '.join(s['countries']) or 'Unknown'})"
                                 for s in rollup)
    system_msg = ("You are a defense supply chain risk analyst supporting the "
                  "Indian Army. Return ONLY a single valid JSON object — no "
                  "markdown, no commentary.")
    user_msg = textwrap.dedent(f"""
    SCENARIO: {scenario}

    SUPPLIERS IN PORTFOLIO:
    {names_countries}

    Return JSON with EXACTLY this structure:
    {{
      "scenario_summary": "<1-2 sentence summary>",
      "crisis_countries": ["<country>"],
      "executive_summary": "<2-3 sentences for the Indian Army>",
      "critical_alerts": ["<alert>", "..."],
      "immediate_actions": ["<action>", "..."]
    }}
    """).strip()

    for attempt in range(1, 3):
        try:
            raw = llm_fn(system_msg, user_msg, max_new_tokens=500)
            result = _parse_llm_output(raw)
            if result.get("executive_summary"):
                return result
            raise ValueError("missing 'executive_summary'")
        except Exception as e:
            print(f"    ⚠️ Framing call attempt {attempt}/2 failed: {e}")

    return {
        "scenario_summary": scenario, "crisis_countries": [],
        "executive_summary": ("LLM framing unavailable — narrative fields are "
                               "placeholders; per-supplier scores below are still live."),
        "critical_alerts": ["LLM framing call failed — review manually."],
        "immediate_actions": ["Re-run this cell — the deterministic score above is unaffected."],
    }

def _batch_call(llm_fn, scenario: str, batch: list) -> dict:
    """Scores ONE small batch of suppliers. Falls back to deterministic
    baseline scoring for just this batch if the LLM fails 3 times, so a
    slow/bad batch never blanks out the rest of the portfolio."""
    system_msg = textwrap.dedent("""
    You are a defense supply chain risk analyst supporting the Indian Army.
    Each supplier already has a deterministic baseline_country_risk (0-100)
    and soe_flag from ownership records — treat these as ground truth.
    Blend the baseline with scenario-specific reasoning to produce a final
    risk_score (0-100) per supplier. Recommend one realistic alternate for
    each supplier scoring >= 60, strongly preferring Indian (Make in India)
    or allied-nation suppliers, and give a one-word
    component_category (e.g. "Vehicle_part").
    List EACH supplier below EXACTLY ONCE. Return ONLY valid JSON, no prose.
    """).strip()
    rollup_summary = "\n".join(_rollup_line(s) for s in batch)
    user_msg = textwrap.dedent(f"""
    SCENARIO: {scenario}

    SUPPLIER BATCH:
    {rollup_summary}

    Return JSON with EXACTLY this structure:
    {{
      "suppliers": [
        {{"name": "<exact name from batch>", "risk_score": <int 0-100>,
          "risk_reason": "<1 sentence citing scenario impact>"}}
      ],
      "alternate_suppliers": [
        {{"replaces": "<at-risk supplier>", "name": "<alternate company>",
          "country": "<country>", "component_category": "<short category>",
          "components_supplied": "<what it makes>", "reason": "<why>"}}
      ]
    }}
    Score every supplier in this batch, exactly once each.
    """).strip()

    for attempt in range(1, 4):
        try:
            raw = llm_fn(
                system_msg,
                user_msg if attempt == 1 else
                (f"Scenario: {scenario}\nReturn ONLY valid JSON matching the schema "
                 f"above, scoring exactly: {', '.join(s['name'] for s in batch)}"),
                max_new_tokens=700,
            )
            result = _parse_llm_output(raw)
            if result.get("suppliers"):
                return result
            raise ValueError("missing 'suppliers' key")
        except Exception as e:
            print(f"    ⚠️ Batch attempt {attempt}/3 failed: {e}")

    print("     ❌ Batch LLM exhausted — using deterministic baseline for this batch.")
    return {
        "suppliers": [{"name": s["name"], "risk_score": int(s["baseline_country_risk"]),
                        "risk_reason": "Baseline country-risk score (LLM overlay unavailable)."}
                       for s in batch],
        "alternate_suppliers": [],
    }

def analyze_scenario(llm_fn, scenario: str, rollup: list) -> dict:
    print("  📡 Framing call (scenario summary, alerts, actions)...")
    framing = _framing_call(llm_fn, scenario, rollup)

    batches = [rollup[i:i + SUPPLIER_BATCH_SIZE]
               for i in range(0, len(rollup), SUPPLIER_BATCH_SIZE)]
    all_suppliers, all_alts = [], []
    for i, batch in enumerate(batches, 1):
        print(f"  📡 Scoring batch {i}/{len(batches)} "
              f"({len(batch)} suppliers)...")
        res = _batch_call(llm_fn, scenario, batch)
        all_suppliers.extend(res.get("suppliers", []))
        all_alts.extend(res.get("alternate_suppliers", []))

    return {**framing, "suppliers": all_suppliers, "alternate_suppliers": all_alts}

# ── CELL 15: Enrich LLM supplier list with ownership + GRAPH data ─
def enrich_suppliers(analysis: dict, rollup: list) -> list:
    """Merges the LLM's per-supplier scores with deterministic GRAPH/
    ownership data. De-duplicates by supplier name FIRST — if the LLM
    repeated a name (small models sometimes do), only the entry with
    the higher risk_score is kept, and a warning is printed. This is
    what prevents duplicate rows downstream (e.g. in the Ownership
    Analysis table)."""
    rollup_by_name = {s["name"]: s for s in rollup}

    deduped: dict = {}
    dup_count = 0
    for sup in analysis.get("suppliers", []):
        name = sup.get("name")
        if not name:
            continue
        if name in deduped:
            dup_count += 1
            if sup.get("risk_score", 0) > deduped[name].get("risk_score", 0):
                deduped[name] = sup
        else:
            deduped[name] = sup
    if dup_count:
        print(f"  ⚠️  LLM repeated {dup_count} supplier name(s) — de-duplicated "
              f"(kept the higher score for each).")

    enriched = []
    for name, sup in deduped.items():
        base = rollup_by_name.get(name, {})
        enriched.append({
            **sup,
            "country":   ", ".join(base.get("countries",  ["Unknown"])),
            "component": ", ".join(base.get("components", ["?"])),
            "vehicle":   ", ".join(base.get("vehicles",   ["?"])),
            "tier":      ", ".join(base.get("tiers",      ["?"])),
            "soe_flag":          base.get("soe_flag", False),
            "total_state_pct":   base.get("total_state_pct", 0.0),
            "baseline_country_risk": base.get("baseline_country_risk", 30.0),
            "is_sole_source": False,
        })
    return enriched

# ── CELL 16: Fill missing alternates from INDIAN_ALTERNATES_CATALOG
def fill_alternates(analysis: dict, enriched_suppliers: list) -> list:
    alts = list(analysis.get("alternate_suppliers", []))
    covered = {a["replaces"] for a in alts}
    by_name = {s["name"]: s for s in enriched_suppliers}

    for s in enriched_suppliers:
        if s["name"] not in covered:
            alt = _indian_alternate(s["component"], s["name"])
            entry = {
                "replaces":           s["name"],
                "name":               alt["name"],
                "country":            alt["country"],
                "component_category": alt.get("component_category", "—"),
                "components_supplied":alt["components_supplied"] or s["component"],
                "reason":             "GeoGRAPHically secure substitute with existing capacity.",
            }
            entry.update(readiness_score(alt, s))
            alts.append(entry)
        else:
            for a in alts:
                if a["replaces"] == s["name"]:
                    alt = _indian_alternate(s["component"], s["name"])
                    if "component_category" not in a:
                        a["component_category"] = alt.get("component_category", "—")
                    if "readiness_score" not in a:
                        a.update(readiness_score(alt, s))
    return alts

# ── CELL 17: Country / scenario selection ─────────────────────
# NOTE: this cell calls Python's input() — it pauses
CRISIS_COUNTRY_OVERRIDE = None

raw_countries = {
    edge[k].strip()
    for edge in GRAPH
    for k in ("t1_country", "t2_country")
    if edge[k].strip().lower() not in ("unknown", "nan", "")
}
raw_countries |= {
    e["country"].strip()
    for v in OWNERSHIP_REGISTRY.values()
    for e in v
    if e["country"].strip().lower() not in ("unknown", "nan", "")
}
available_countries = sorted(raw_countries)

print("\n Countries found in your data:")
for i, c in enumerate(available_countries, 1):
    print(f"  [{i}] {c}")

if CRISIS_COUNTRY_OVERRIDE:
    selection = CRISIS_COUNTRY_OVERRIDE.strip()
    print(f"\n(Using CRISIS_COUNTRY_OVERRIDE = {selection!r} — skipping input() prompt.)")
else:
    selection = input(
        "\nEnter a country name or number to focus on as the Crisis Zone\n"
        "(or type a free-text scenario and press Enter): "
    ).strip()

if selection.isdigit() and 1 <= int(selection) <= len(available_countries):
    CRISIS_COUNTRY = available_countries[int(selection) - 1]
    SCENARIO = f"Geopolitical crisis / conflict escalation involving {CRISIS_COUNTRY}"
else:
    matched = [c for c in available_countries if c.lower() == selection.lower()]
    if matched:
        CRISIS_COUNTRY = matched[0]
        SCENARIO = f"Geopolitical crisis / conflict escalation involving {CRISIS_COUNTRY}"
    else:
        SCENARIO = selection or "Generalised geopolitical instability across major supplier countries"
        CRISIS_COUNTRY = "Multiple"

print(f"\n🚨 Crisis country : {CRISIS_COUNTRY}")
print(f"   Scenario       : {SCENARIO}")

# ── CELL 18: Run full AI pipeline ─────────────────────────────
def _recompute_portfolio_score(enriched: list) -> tuple:
    """Deterministically derive the headline portfolio score/level from
    the FINAL, de-duplicated, enriched per-supplier scores — i.e. from
    exactly what the dashboard plots. This guarantees the header can
    never disagree with the chart, regardless of what the LLM said
    about itself."""
    scores = [s.get("risk_score", 0) for s in enriched]
    if not scores:
        return 0, "LOW"
    avg = round(sum(scores) / len(scores))
    level = ("CRITICAL" if avg >= 70 else
              "HIGH"     if avg >= 50 else
              "MEDIUM"   if avg >= 30 else "LOW")
    return avg, level

def run_agent(scenario: str, crisis_country: str):
    print("=" * 70)
    print(f"  🇮🇳 {AGENT_TITLE.upper()}")
    print(f"  LLM Backend     : {LLM_BACKEND.upper()}")
    print(f"  Ownership depth : ≥{OWNERSHIP_DEPTH_PCT}%")
    print(f"  Suppliers       : {len(supplier_rollup)}")
    print(f"  Scenario        : {scenario}")
    print("=" * 70)

    print(f"\n🤖 Running LLM analysis ({LLM_BACKEND.upper()}) with deterministic grounding...")
    analysis = analyze_scenario(LLM_FN, scenario, supplier_rollup)

    enriched   = enrich_suppliers(analysis, supplier_rollup)   # de-duplicated
    alternates = fill_alternates(analysis, enriched)

    # ── Ownership analysis is 100% deterministic — it reads straight from
    supplier_names = sorted(all_supplier_names)
    ownership_data = run_ownership_analysis(supplier_names, OWNERSHIP_DEPTH_PCT)

    # ── Consistency fix: recompute the headline score from the same
    portfolio_score, overall_level = _recompute_portfolio_score(enriched)
    analysis["portfolio_risk_score"] = portfolio_score
    analysis["overall_risk_level"]   = overall_level

    print(f"  ↳ Portfolio risk   : {analysis['portfolio_risk_score']}/100 — {analysis['overall_risk_level']}  "
          f"(deterministically derived from the {len(enriched)} supplier scores below)")
    print(f"  ↳ Suppliers scored : {len(enriched)}")
    print(f"  ↳ SOE flags        : {sum(1 for o in ownership_data if o['soe_flag'])}")
    print(f"  ↳ Alternates       : {len(alternates)}")

    analysis["suppliers"]           = enriched
    analysis["alternate_suppliers"] = alternates

    # ── Crisis-country source of truth ──────────────────────────
    if crisis_country != "Multiple":
        analysis["crisis_countries"] = [crisis_country]
    elif not analysis.get("crisis_countries"):
        analysis["crisis_countries"] = ["Multiple"]

    return analysis, enriched, alternates, ownership_data

analysis, enriched_suppliers, alternates, ownership_data = run_agent(SCENARIO, CRISIS_COUNTRY)


## GRAPH PLOTS & DASHBOARD GENERATION

In [ ]:
# ── CELL 19: Plotly dashboard ───
# it only reads the analysis
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display, HTML

pio.renderers.default = "colab"

def _rcol(v):
    return "#ef4444" if v >= 70 else "#f59e0b" if v >= 50 else "#22c55e"

def _table(headers, cols, header_color, row_fill):
    return go.Table(
        header=dict(values=[f"<b>{h}</b>" for h in headers], fill_color=header_color,
                    font=dict(color="white", size=10), align="left", height=26),
        cells=dict(values=cols, fill_color=[row_fill] * len(cols),
                   font=dict(size=9), align="left", height=20))

def render_dashboard(analysis: dict, ownership_data: list):
    suppliers  = sorted(analysis["suppliers"], key=lambda x: x.get("risk_score", 0), reverse=True)
    alternates = analysis["alternate_suppliers"]
    score      = analysis["portfolio_risk_score"]
    level      = analysis["overall_risk_level"]
    crisis     = ", ".join(analysis.get("crisis_countries", ["Multiple"]))

    # ── Crisis exposure detection ───────────────────────────────
    crisis_countries_list = analysis.get("crisis_countries", [crisis])
    crisis_norm = {normalize_country(cc) for cc in crisis_countries_list if normalize_country(cc)}

    def _supplier_countries(s):
        return [c.strip() for c in s.get("country", "").split(",") if c.strip()]

    exposure_by_supplier: dict = {}   # name -> {"types": set, "detail": [...], "crisis_owners": [...]}
    for s in suppliers:
        name = s["name"]
        types, detail = set(), []

        # (1) Direct location in the crisis country
        if any(normalize_country(c) in crisis_norm for c in _supplier_countries(s)):
            types.add("Direct")
            detail.append(f"Located in crisis country ({crisis}).")

        # (2) Tier-2 dependency propagation (this supplier is a Tier-1 whose
        # Tier-2 source sits in the crisis country)
        dep_pairs = TIER1_TO_TIER2_COUNTRIES.get(name, set())
        crisis_t2 = sorted({t2 for t2, t2c in dep_pairs if normalize_country(t2c) in crisis_norm})
        if crisis_t2:
            types.add("Tier-2 Dependency")
            detail.append(f"Depends on Tier-2 supplier(s) located in {crisis}: {', '.join(crisis_t2)}.")

        # (3) Ownership exposure — a qualifying owner based in the crisis country
        owners = OWNERSHIP_REGISTRY.get(normalize_name(name), [])
        crisis_owners = [o for o in owners
                          if normalize_country(o["country"]) in crisis_norm
                          and o["stake_pct"] >= OWNERSHIP_DEPTH_PCT]
        if crisis_owners:
            types.add("Ownership-Linked")
            stake_str = ", ".join(f"{o['owner']} ({o['stake_pct']}%)" for o in crisis_owners)
            detail.append(f"Owner(s) based in {crisis} hold a qualifying stake: {stake_str}.")

        if types:
            exposure_by_supplier[name] = {"types": types, "detail": detail, "crisis_owners": crisis_owners}
    # NOTE: deliberately NO "if empty, show all suppliers" fallback. An empty

    # ── Deterministic risk-formula boost for every exposed supplier ──────
    crisis_suppliers_bar = []
    for s in suppliers:
        exp = exposure_by_supplier.get(s["name"])
        if not exp:
            continue

        base_weight = s.get("baseline_country_risk", 30.0)
        geo_signal  = (20 if "Direct" in exp["types"] else
                       14 if "Tier-2 Dependency" in exp["types"] else 0)
        own_signal  = 0.0
        if exp["crisis_owners"]:
            total_stake = sum(o["stake_pct"] for o in exp["crisis_owners"])
            # A qualifying stake contributes a baseline exposure score even
            # at the minimum threshold (e.g. 1%), then scales up with size.
            own_signal = min(15.0, round(5 + total_stake * 0.5, 1))
        fin_signal   = financial_distress_score(s["name"])
        conc_penalty = concentration_penalty(s["name"])

        deterministic_score = min(100, round(
            base_weight + geo_signal + own_signal + fin_signal + conc_penalty))

        boosted = dict(s)   # never mutate the original enriched-supplier dict
        if deterministic_score > boosted.get("risk_score", 0):
            existing_reason = boosted.get("risk_reason", "").strip()
            boosted["risk_reason"] = (existing_reason + " " + " ".join(exp["detail"])).strip()
            boosted["risk_score"] = deterministic_score
        boosted["exposure_type"] = " + ".join(sorted(exp["types"]))
        crisis_suppliers_bar.append(boosted)

    crisis_suppliers_bar.sort(key=lambda x: x.get("risk_score", 0), reverse=True)

    # Apply the same boosted scores back onto the full supplier list
    boosted_by_name = {s["name"]: s for s in crisis_suppliers_bar}
    suppliers = sorted(
        [boosted_by_name.get(s["name"], s) for s in suppliers],
        key=lambda x: x.get("risk_score", 0), reverse=True,
    )
    if crisis_suppliers_bar:
        score = round(sum(s.get("risk_score", 0) for s in suppliers) / len(suppliers))
        level = ("CRITICAL" if score >= 70 else "HIGH" if score >= 50 else
                  "MEDIUM" if score >= 30 else "LOW")

    crisis_supplier_names = set(exposure_by_supplier.keys())
    crisis_alts = [a for a in alternates if a.get("replaces") in crisis_supplier_names]

    # Ownership rows for crisis-zone suppliers — covers suppliers pulled in
    crisis_ownership = [o for o in ownership_data if o["name"] in crisis_supplier_names]

    # Defensive de-dupe: ownership_data should already be unique (Cell 10
    seen_supplier = set()
    ownership_data_unique = []
    for o in ownership_data:
        if o["name"] in seen_supplier:
            continue
        seen_supplier.add(o["name"])
        ownership_data_unique.append(o)
    ownership_data = ownership_data_unique

    # Also de-dupe the already-filtered crisis_ownership list
    seen_crisis = set()
    crisis_ownership_unique = []
    for o in crisis_ownership:
        if o["name"] in seen_crisis:
            continue
        seen_crisis.add(o["name"])
        crisis_ownership_unique.append(o)
    crisis_ownership = crisis_ownership_unique

    soe_flagged = [o for o in crisis_ownership if o["soe_flag"]]

    # ── Diagnostic: always surface the match results ───────────
    print(f"\n📊 Dashboard filter diagnostics:")
    print(f"   Crisis country norm-keys : {crisis_norm}")
    print(f"   Suppliers in crisis zone : {len(crisis_suppliers_bar)}"
          + (f"  → {[(s['name'], s['exposure_type']) for s in crisis_suppliers_bar]}"
             if crisis_suppliers_bar else
             "  ← 0 MATCHED — check that the country name in your CSV exactly matches what's shown above"))
    print(f"   Crisis-zone alts found   : {len(crisis_alts)}")
    print(f"   Ownership rows (crisis)  : {len(crisis_ownership)}")
    print(f"   SOE flags (crisis-scoped): {len(soe_flagged)}  "
          f"(portfolio-wide SOE total, for reference: {sum(1 for o in ownership_data if o['soe_flag'])})")

    fig = make_subplots(
        rows=4, cols=1,
        subplot_titles=(
            f"📊 Supplier Risk Scores — Crisis: {crisis}",
            "🚛 Full Supplier Risk Register",
            f"🏛️ Ownership Analysis — Stakes ≥{OWNERSHIP_DEPTH_PCT}%",
            "🔷 Alternate Suppliers",
        ),
        specs=[
            [{"type": "bar"}],
            [{"type": "table"}],
            [{"type": "table"}],
            [{"type": "table"}],
        ],
        vertical_spacing=0.06,
        row_heights=[0.20, 0.22, 0.30, 0.28],
    )

    # Row 1a — Supplier Risk Scores (crisis-zone suppliers: direct, tier-2
    # dependency, or ownership-linked — see exposure_by_supplier above)
    if crisis_suppliers_bar:
        fig.add_trace(go.Bar(
            x=[s["name"] for s in crisis_suppliers_bar],
            y=[s.get("risk_score", 0) for s in crisis_suppliers_bar],
            marker_color=[_rcol(s.get("risk_score", 0)) for s in crisis_suppliers_bar],
            text=[str(s.get("risk_score", 0)) for s in crisis_suppliers_bar],
            textposition="outside", showlegend=False,
            customdata=[s.get("exposure_type", "") for s in crisis_suppliers_bar],
            hovertemplate="<b>%{x}</b><br>Risk: %{y}/100<br>Exposure: %{customdata}<extra></extra>",
        ), row=1, col=1)
        fig.add_hline(y=RISK_ALERT_THRESHOLD, line_dash="dash", line_color="#f59e0b",
                      annotation_text=f"Alert Threshold ({RISK_ALERT_THRESHOLD})", row=1, col=1)
    else:
        # Empty-state: render a placeholder bar with a visible annotation
        # rather than a blank / invisible axis.
        fig.add_trace(go.Bar(
            x=["(no suppliers matched)"], y=[0],
            marker_color=["#cbd5e1"], showlegend=False,
        ), row=1, col=1)
        fig.add_annotation(
            text=(f"No suppliers in your data are tied to: {crisis}<br>"
                  f"(checked direct location, Tier-2 dependency, and ownership links — "
                  f"verify the country name in your CSVs matches exactly)"),
            xref="x domain", yref="y domain", x=0.5, y=0.5,
            showarrow=False, font=dict(size=11, color="#64748b"),
            align="center", row=1, col=1,
        )

    # Row 2 — Supplier Risk Register
    def _status(v):
        return "🔴 CRITICAL" if v >= 70 else "🟡 MEDIUM" if v >= 50 else "🟢 LOW"

    fig.add_trace(_table(
        ["Supplier", "Country", "Component", "Vehicle", "Tier", "Risk Score",
         "Reason", "Crisis Exposure", "Status"],
        [
            [s["name"]                          for s in suppliers],
            [s.get("country", "?")               for s in suppliers],
            [s.get("component", "?")             for s in suppliers],
            [s.get("vehicle", "?")               for s in suppliers],
            [s.get("tier", "?")                  for s in suppliers],
            [s.get("risk_score", 0)              for s in suppliers],
            [s.get("risk_reason", "")            for s in suppliers],
            [s.get("exposure_type", "—")         for s in suppliers],
            [_status(s.get("risk_score",0))      for s in suppliers],
        ],
        "#1e293b",
        ["#fee2e2" if s.get("risk_score",0)>=70 else "#fef9c3" if s.get("risk_score",0)>=50 else "#dcfce7"
         for s in suppliers],
    ), row=2, col=1)

    # Row 3 — Ownership Analysis (crisis-country suppliers only, per-owner rows)
    own_rows = []
    for o in crisis_ownership:
        for ow in o["owners"]:
            is_soe = (o["soe_flag"] and (
                ow["type"] in STATE_TYPES
                or SOE_KEYWORDS.search(str(ow["type"]))
                or SOE_KEYWORDS.search(str(ow["owner"]))
            ))
            own_rows.append({
                "supplier": o["name"],
                "owner":    ow["owner"],
                "type":     ow["type"],
                "country":  ow["country"],
                "stake":    ow["stake_pct"],
                "board":    ow.get("board_role", ""),
                "soe":      "🚨 SOE" if is_soe else "",
            })

    if own_rows:
        fig.add_trace(_table(
            ["Supplier", "Owner", "Type", "Owner Country",
             f"Stake % (≥{OWNERSHIP_DEPTH_PCT}%)", "Board Role / SOE Flag"],
            [
                [r["supplier"]                          for r in own_rows],
                [r["owner"]                             for r in own_rows],
                [r["type"]                              for r in own_rows],
                [r["country"]                           for r in own_rows],
                [f"{r['stake']}%"                       for r in own_rows],
                [f"{r['board']}  {r['soe']}".strip()    for r in own_rows],
            ],
            "#7c3aed",
            ["#fde8ff" if r["soe"] else "#f0fdf4" for r in own_rows],
        ), row=3, col=1)
    else:
        # Render a single-row placeholder so the panel is never blank/invisible
        fig.add_trace(_table(
            ["Supplier", "Owner", "Type", "Owner Country",
             f"Stake % (≥{OWNERSHIP_DEPTH_PCT}%)", "Board Role / SOE Flag"],
            [["—"], ["No ownership records for crisis-zone suppliers"],
             ["—"], ["—"], ["—"], ["—"]],
            "#7c3aed", ["#f8f9fa"],
        ), row=3, col=1)

    # Row 4 — Alternate Suppliers (with component_category column)
    # (alternates that replace a crisis-zone Tier-1/Tier-2 supplier)
    # Cap = alternate_supplier_count (config.yaml) × number of at-risk
    # suppliers actually in the crisis zone, floor 12 so small crises
    # still show a readable table.
    _alt_cap = max(12, ALT_SUPPLIER_COUNT * max(len(crisis_supplier_names), 1))
    top12 = crisis_alts[:_alt_cap]
    if top12:
        fig.add_trace(_table(
            ["Replaces", "Alternate Supplier", "Country", "Category",
             "Components Supplied", "Readiness", "Reason"],
            [
                [a.get("replaces", "")              for a in top12],
                [a.get("name", "")                  for a in top12],
                [a.get("country", "")               for a in top12],
                [a.get("component_category", "—")   for a in top12],
                [a.get("components_supplied", "")   for a in top12],
                [f"{a.get('readiness_score', '—')}/100" for a in top12],
                [a.get("reason", "")                for a in top12],
            ],
            "#0f766e",
            ["#f0fdf4"] * len(top12),
        ), row=4, col=1)
    else:
        # Empty-state placeholder — no silent fallback to the full
        # alternates catalog, consistent with the rest of the dashboard.
        fig.add_trace(_table(
            ["Replaces", "Alternate Supplier", "Country", "Category",
             "Components Supplied", "Readiness", "Reason"],
            [["—"], ["No alternates mapped to crisis-zone suppliers"],
             ["—"], ["—"], ["—"], ["—"], ["—"]],
            "#0f766e", ["#f8f9fa"],
        ), row=4, col=1)

    fig.update_layout(
        title=dict(
            text=(f"🇮🇳 {AGENT_TITLE.upper()}<br><sup>"
                  f"Scenario: {analysis.get('scenario_summary','')[:90]}  |  "
                  f"Crisis: {crisis}  |  Risk: {score}/100 — {level}  |  "
                  f"LLM: {LLM_BACKEND.upper()}  |  "
                  f"{datetime.datetime.now():%Y-%m-%d %H:%M} UTC</sup>"),
            font=dict(size=14), x=0.5, xanchor="center",
        ),
        height=1800, paper_bgcolor="#f8fafc",
        font=dict(family="Inter, sans-serif", size=10),
        margin=dict(t=100, b=40, l=40, r=60),
    )
    fig.update_xaxes(tickangle=-35, row=1, col=1)

    # ── Render: try the live widget renderer, then GUARANTEE a fallback ──
    try:
        fig.show()
    except Exception as e:
        print(f"⚠️  Live renderer failed ({e}); falling back to embedded HTML.")
    # Always also embed a static HTML version — this is what fixes the
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))

    # ── HTML KPI banner ───────────────────────────────────────
    alerts_html  = "".join(
        f'<div style="background:#fee2e2;border-left:4px solid #ef4444;padding:6px 10px;'
        f'margin:3px 0;border-radius:4px;font-size:11px;">🚨 {a}</div>'
        for a in analysis.get("critical_alerts", []))
    actions_html = "".join(
        f"<li style='margin:2px 0;font-size:11px;'>{a}</li>"
        for a in analysis.get("immediate_actions", []))
    soe_html = "".join(
        f'<div style="background:#fde8ff;border-left:4px solid #7c3aed;padding:6px 10px;'
        f'margin:3px 0;border-radius:4px;font-size:11px;">⚠️ {o["name"]} — '
        f'State stake: {o["total_state_pct"]}% — SOE FLAG: {"YES 🚨" if o["soe_flag"] else "Watch"}</div>'
        for o in crisis_ownership if o["state_owners"])
    # Deterministic recount — never trust the LLM's narrative alert list for
    # the KPI number; recompute from the exact scores being plotted, using
    # the config.yaml-driven risk_threshold cutoff.
    alert_count = sum(
        1 for s in crisis_suppliers_bar if s.get("risk_score", 0) >= RISK_ALERT_THRESHOLD
    )

    kpi_cards = "".join(
        f'<div style="background:#1e293b;padding:9px 16px;border-radius:8px;min-width:120px;text-align:center;">'
        f'<div style="font-size:9px;color:#94a3b8;text-transform:uppercase;">{label}</div>'
        f'<div style="font-size:22px;font-weight:700;color:{color};">{value}</div></div>'
        for label, value, color in [
            ("Portfolio Risk", f"{score}/100",   "#ef4444" if score >= 70 else "#f59e0b"),
            ("Risk Level",     level,             "#ef4444" if level == "CRITICAL" else "#f59e0b"),
            ("Suppliers",      len(suppliers),    "#60a5fa"),
            ("Critical Alerts",alert_count,       "#ef4444"),
            ("Alternates",     len(crisis_alts),  "#22c55e"),
            ("LLM",            LLM_BACKEND.upper(),"#818cf8"),
            ("Own. Depth",     f"≥{OWNERSHIP_DEPTH_PCT}%", "#f59e0b"),
        ]
    )
    display(HTML(f"""
    <div style="font-family:Inter,sans-serif;padding:14px;background:#0f172a;
                color:white;border-radius:10px;margin:10px 0;">
      <h2 style="margin:0 0 10px;font-size:15px;">🛡️ {AGENT_TITLE}
        &nbsp;<span style="font-size:11px;color:#64748b;">Crisis: {crisis}
        &nbsp;|&nbsp;{LLM_BACKEND.upper()} powered</span></h2>
      <div style="display:flex;gap:10px;flex-wrap:wrap;margin-bottom:10px;">{kpi_cards}</div>
      <p style="font-size:11px;color:#cbd5e1;margin:6px 0 10px;">{analysis.get("executive_summary","")}</p>
      <div style="display:flex;gap:10px;flex-wrap:wrap;">
        <div style="flex:1;min-width:240px;background:#1e293b;padding:10px;border-radius:8px;">
          <div style="color:#f87171;font-size:10px;font-weight:700;margin-bottom:4px;">🔴 CRITICAL ALERTS</div>
          {alerts_html or '<div style="font-size:11px;color:#94a3b8;">None triggered</div>'}
        </div>
        <div style="flex:1;min-width:240px;background:#1e293b;padding:10px;border-radius:8px;">
          <div style="color:#4ade80;font-size:10px;font-weight:700;margin-bottom:4px;">✅ IMMEDIATE ACTIONS</div>
          <ul style="margin:0;padding-left:14px;">{actions_html}</ul>
        </div>
        <div style="flex:1;min-width:240px;background:#1e293b;padding:10px;border-radius:8px;">
          <div style="color:#c084fc;font-size:10px;font-weight:700;margin-bottom:4px;">
            🏛️ OWNERSHIP FLAGS (≥{OWNERSHIP_DEPTH_PCT}%)</div>
          {soe_html or '<div style="font-size:11px;color:#94a3b8;">No state-linked owners above threshold</div>'}
        </div>
      </div>
    </div>"""))
    return fig

print("\n📊 Rendering dashboard...")
render_dashboard(analysis, ownership_data)

output_package = {
        "scenario": SCENARIO, "analysis": analysis,
        "suppliers": enriched_suppliers, "alternates": alternates,
    }
with open("dashboard_output.json", "w", encoding="utf-8") as f:
    _json.dump(output_package, f, indent=2, default=str)
print("\n✅ dashboard_output.json saved")